# LLM Fundamentals coding seminar.

Welcome to the LLM Fundamentals Coding Seminar! This session is designed to give you hands-on experience working with LLMs. You will practice such components as data preparation, tokenization, model decoding, and attention mechanisms implementation using modern libraries like Hugging Face Transformers and PyTorch.

In [ ]:
!pip install transformers==4.56.1 torch==2.8.0

In [ ]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import random
import numpy as np
import typing as tp

In [ ]:
def set_seed(seed: int = 42):
    """
    Fix all random seeds for reproducibility.
    """
    # Python built-in random
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch (CPU and CUDA)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # all GPUs

    # Ensure deterministic behavior (may impact performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Random seed set to {seed}")


seed = 2025
set_seed(seed)

Random seed set to 2025


### Task 1: LLM Pretraining Data Preparation

Write a function that prepares data for a **causal language modeling** task.

Your function should:

1. Take a list of tokens as input.  
2. For every token position `i` (starting from 1 - follow up: why not from 0?), create a pair:  
   - **Context**: all tokens from the beginning up to (but not including) position `i`.  
   - **Target**: the token at position `i`.  
3. Return a list of `(context, target)` pairs.

---

**Example:**

```python
tokens = ["The", "cat", "sat"]

make_contextual_lm_data(tokens)
# [
#   (["The"], "cat"),
#   (["The", "cat"], "sat")
# ]

In [ ]:
def make_contextual_lm_data(tokens: tp.List) -> tp.List[tp.Tuple]:
    """
    For each position in the sequence, show how the model
    uses all previous tokens as context to predict the next one.

    Returns:
        List of tuples: [(context_list, target_token), ...]
    """
    output = []
    for i in range(1, len(tokens)):
      context = tokens[:i]
      target = tokens[i]
      output.append((context, target))
    return output

In [ ]:
tokens = ["The", "cat", "sat", "on", "the", "mat"]
result = make_contextual_lm_data(tokens)

EXPECTED_RESULT = [
    (["The"], "cat"),
    (["The", "cat"], "sat"),
    (["The", "cat", "sat"], "on"),
    (["The", "cat", "sat", "on"], "the"),
    (["The", "cat", "sat", "on", "the"], "mat"),
]

assert result == EXPECTED_RESULT

### Task 2: Tokenization with GPT-2

Write a function that demonstrates how text is converted into tokens using a pretrained GPT-2 tokenizer.

Your function should:

1. Take a string of text as input.  
2. Use the GPT-2 tokenizer to produce a sequence of **token IDs**.  
3. Convert each token ID back into its corresponding **string token** for easier inspection.  
4. Return both the list of token IDs and the list of string tokens.
---

Original GPT-2 tokenizer implementation can be found [here.](https://github.com/openai/gpt-2/blob/master/src/encoder.py)


**Example:**

```python
text = "The cat sat"
token_ids, tokens = tokenize_fn(text)

# token_ids → [464, 818, 3797]
# tokens    → ["The", " cat", " sat"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")


def tokenize_fn(text: str) -> tp.Tuple:
    """
    Tokenize text using GPT-2 tokenizer.
    Returns both token IDs and string tokens.
    """
    token_ids = tokenizer.encode(text)
    tokens = [tokenizer.decode([t_id]) for t_id in token_ids]
    return token_ids, tokens


In [ ]:
def test_tokenize_fn(text):

    # Expected GPT-2 tokenization
    gold_tokens = [
        "Del",
        "icious",
        " past",
        "ries",
        " and",
        " cro",
        "iss",
        "ants",
        " attract",
        " many",
        " visitors",
        ".",
    ]
    gold_ids = [13856, 6243, 1613, 1678, 290, 6763, 747, 1187, 4729, 867, 9692, 13]

    token_ids, tokens = tokenize_fn(text)

    assert tokens == gold_tokens, f"Expected tokens {gold_tokens}, but got {tokens}"
    assert token_ids == gold_ids, f"Expected IDs {gold_ids}, but got {token_ids}"

    return "Test passed!"


# Run test
text = "Delicious pastries and croissants attract many visitors."

test_tokenize_fn(text)

'Test passed!'

## Task 3: Apply Chat Template to a Conversational LLM

Modern conversational LLMs are usually trained on top of base pretrained LLMs using **chat templates**.  
A chat template defines how messages from different roles (system, user, assistant) are formatted into a single input sequence for the model.  
This helps the LLM behave conversationally, by giving it explicit role markers and structured input.

Your function should:

1. Take a user prompt as input.  
2. Wrap the prompt in a **chat message format** (e.g., `{"role": "user", "content": prompt}`).  
3. Apply the model’s built-in **chat template**.  
4. Convert the formatted text into token IDs and return the model inputs (ready for inference) as pytorch tensors. Use ```tokenizer(...)``` callable.



In [ ]:
model_id = "Qwen/QwQ-32B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- '' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
  {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if (message.role == "user") or (message.role == "system" and not loop.first) %}
    

In [ ]:
def tokenize_with_chat_template(prompt: str):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    input_ids = tokenizer(text, return_tensors="pt")
    return input_ids


In [ ]:
prompt = "Describe London in one sentence"

model_inputs = tokenize_with_chat_template(prompt)

parsed_output = tokenizer.decode(model_inputs["input_ids"][0]).strip("\n")

EXPECTED_OUTPUT = """
<|im_start|>user
Describe London in one sentence<|im_end|>
""".strip()

assert parsed_output == EXPECTED_OUTPUT

## Task 4: Using the `add_generation_prompt` Flag

**Goal:** Explore the `add_generation_prompt` flag in `tokenize_with_chat_template`.  

**Exercise:**  
- Experiment with the flag set to `True` and `False`.  
- Observe and compare how the tokenized input changes.  
- Think about when this flag might be useful or unnecessary.



In [ ]:
def tokenize_with_chat_template(prompt: str, **params):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, **params)
    input_ids = tokenizer(text, return_tensors="pt")
    return input_ids

In [ ]:
prompt = "Describe London in one sentence"

params = {
    "tokenize": False,
    "add_generation_prompt": True,
}

model_inputs = tokenize_with_chat_template(prompt, **params)

In [ ]:
parsed_output = tokenizer.decode(model_inputs["input_ids"][0]).strip("\n")

EXPECTED_OUTPUT = """
<|im_start|>user
Describe London in one sentence<|im_end|>
<|im_start|>assistant
<think>
""".strip()
assert parsed_output == EXPECTED_OUTPUT

### Task 5: Generate Text Completion Using Greedy Decoding (Native Transformers)

**Goal:**  
Use the Hugging Face Transformers library to perform **greedy text generation** without writing a manual decoding loop.

**Description:**  
- Use the model’s built-in `generate` method with settings that enforce greedy decoding.
- Set max_new_tokens to 20.
- Provide an initial prompt and generate a text completion. Make sure the completion is reproducible!
- Return output_ids as list of tokens (make sure to remove tokens that are not part of the completion)

In [ ]:
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

In [ ]:
prompt = "Describe London in one sentence"

params = {
    "tokenize": False,
    "add_generation_prompt": True,
    "enable_thinking": False,
}

model_inputs = tokenize_with_chat_template(prompt, **params)


def generate_with_built_in(model):

    generated_ids = model.generate(**model_inputs,
                                   max_new_tokens=20,
                                   temperature=0.0,
                                   do_sample=False,
                                   top_k=None)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
    return output_ids


output_ids = generate_with_built_in(model)

In [ ]:
parsed_output = tokenizer.decode(output_ids, skip_special_tokens=False).strip("\n")

assert (
    parsed_output
    == "London is a city in England, known for its historic landmarks, cultural diversity, and vibrant atmosphere."
)

### Task 6: Implement Standard Greedy LLM Decoding

**Goal:**  
Implement a simple greedy decoding function for a language model. Greedy decoding selects the token with the highest probability at each step until the end-of-sequence token or a maximum length is reached.

**Description:**  
- The function should take a model, an initial input tensor of token IDs, and a maximum number of tokens to generate.  
- At each step, compute the model’s logits for the last token, pick the token with the highest probability, append it to the sequence, and continue.  
- Stop generation if the model produces the end-of-sequence token.


![](https://miro.medium.com/1*QLA1jwTY0HBBpLBWfw_QaQ.png)

Next token prediction illustration. [Source.](https://medium.com/@lmpo/mastering-llms-a-guide-to-decoding-algorithms-c90a48fd167b)

In [ ]:
def generate(model, ids, max_tokens):
    "Implement manual greedy decoding"

    for i in range(max_tokens):

        with torch.no_grad():
            logits = model(ids).logits

        logits = logits[:, -1, :]

        idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        ids = torch.cat([ids, idx_next], dim=-1)

        if idx_next.item() == tokenizer.eos_token_id:
            break

    return ids


In [ ]:
prompt = "Describe London in one sentence"

max_tokens = 20
generated_ids = generate(model, model_inputs["input_ids"], max_tokens=max_tokens)

output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()

In [ ]:
parsed_output = tokenizer.decode(output_ids, skip_special_tokens=False).strip("\n")

assert (
    parsed_output
    == "London is a city in England, known for its historic landmarks, cultural diversity, and vibrant atmosphere."
)

### Task 7: Add Temperature Scaling to the Generate Function

**Goal:**  
Extend the ```generate``` function to support **temperature scaling** for more flexible token sampling.

**Description:**  
- The `temperature` parameter adjusts the “confidence” of the model’s predictions:  
  - **Low temperature (close to 0):** model outputs are sharper, similar to greedy decoding.  
  - **High temperature:** model outputs are more diverse, allowing exploration of less probable tokens.  
- When `temperature > 0`, scale the logits by `1/temperature` and sample from the resulting probability distribution instead of always taking the argmax. Use ```torch.multinomial``` for sampling.

![](https://media.licdn.com/dms/image/v2/D5622AQEH98bVz3ly2Q/feedshare-shrink_800/feedshare-shrink_800/0/1717998492049?e=1761177600&v=beta&t=63ZLYBjOvXMnfEpTWMvGFWxnCXZ6-p8GjVDWzF0cops)

*Impact of temperature on a probability distribution. Credit to Harshit Sharma.*

In [ ]:
def generate(model, ids, max_tokens, temperature=0.0):

    for i in range(max_tokens):

        with torch.no_grad():
            logits = model(ids).logits

        logits = logits[:, -1, :]

        if temperature > 0:
            logits /= temperature
            probs = torch.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        ids = torch.cat([ids, idx_next], dim=-1)

        if idx_next.item() == tokenizer.eos_token_id:
            break

    return ids

In [ ]:
prompt = "Describe London in one sentence"

for temp in [0.0, 0.4, 0.8, 1.2]:
    max_tokens = 30
    generated_ids = generate(
        model, model_inputs["input_ids"], max_tokens=max_tokens, temperature=temp
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
    parsed_output = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

    print(f"Temperature={temp}: {parsed_output} \n")

Temperature=0.0: London is a city in England, known for its historic landmarks, cultural diversity, and vibrant atmosphere. 

Temperature=0.4: London is a major city in the United Kingdom, known for its history, culture, and landmarks. 

Temperature=0.8: London is a historic city-centered metropolis in England. 

Temperature=1.2: London is a great city known for its rich history, cultural landmarks, and vibrant atmosphere. 



### Task 8: Add Top-K Sampling to the Generate Function

**Goal:**  
Enhance the generation function to support **top-k sampling**, which limits the token choices to the k most probable tokens at each step.

**Description:**  
- The `top_k` parameter controls how many of the highest-probability tokens are considered for sampling.  
- At each step:  
  - Identify the top `k` logits.  
  - Mask all other logits by setting them to `-inf` (effectively removing them from consideration. Follow up: why does this work?).
  - Sample the next token from the resulting probability distribution.  

![](https://s4.uupload.ir/files/1_ixvvlan_ll3mdldefjj5qa_mma0.png)

[Source.](https://www.kaggle.com/code/sajjadayobi360/how-to-generate-text-using-language-models)

In [ ]:
def generate(model, ids, max_tokens, temperature=0.0, top_k=None):

    for i in range(max_tokens):

        with torch.no_grad():
            logits = model(ids).logits

        logits = logits[:, -1, :]

        if top_k is not None:
            topk, _ = torch.topk(logits, top_k)
            min_val = topk[:, -1]
            logits = torch.where(
                logits > min_val, logits, torch.full_like(logits, float("-inf"))
            )

        if temperature > 0:
            logits /= temperature
            probs = torch.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        ids = torch.cat([ids, idx_next], dim=-1)

        if idx_next.item() == tokenizer.eos_token_id:
            break

    return ids


In [ ]:
prompt = "Describe London in one sentence"

for top_k in [None, 50, 100, 1000]:
    max_tokens = 30
    generated_ids = generate(
        model,
        model_inputs["input_ids"],
        max_tokens=max_tokens,
        temperature=0.6,
        top_k=top_k,
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
    parsed_output = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

    print(f"Top_k={top_k}: {parsed_output} \n")

Top_k=None: London is a vibrant and historic city in England, known for its rich cultural heritage, bustling streets, and iconic landmarks. 

Top_k=50: London is a historic city in England, known for its rich cultural heritage, vibrant streets, and iconic landmarks. 

Top_k=100: London is a city in England known for its historical landmarks, cultural significance, and vibrant city life. 

Top_k=1000: London is a major city in England, known for its historical landmarks, cultural heritage, and vibrant streets. 



### Task 9: Attention implementation


<img src="https://machinelearningmastery.com/wp-content/uploads/2022/03/dotproduct_1.png">

[Source.](https://machinelearningmastery.com/how-to-implement-scaled-dot-product-attention-from-scratch-in-tensorflow-and-keras/)

In [ ]:
class ScaleDotProductAttention(nn.Module):
    """
    A simple implementation of scaled dot-product attention.
    """

    def __init__(self, embed_dim):
        super().__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.scale = embed_dim**0.5

    def forward(self, x):
        """
        x: Tensor of shape (batch, seq_len, embed_dim)
        Returns: attn_output, attn_weights
        """
        batch_size, seq_len, embed_dim = x.size()

        # Project inputs
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Compute attention scores
        # multiply Q and K matrices (remember mat mul rules) and scale the resulting values
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        # Normalizes attention weights
        # use softmax to normalise attention scores, make sure to normalise by rows!
        attn_weights = F.softmax(attn_scores, dim=-1)

        # Copute weighted sum
        # multiply normalised attn weights with V matrix, this is final attn output.
        attn_output = torch.matmul(attn_weights, V)

        # return a tuple of attention output (weighted sum) and attention weights

        return attn_output, attn_weights

### Sytnethic Test Input

We create a simple tensor to test the attention module:

- **Batch size:** `1` – one sequence in the batch.  
- **Sequence length:** `3` – the sequence has three time steps (tokens).  
- **Embedding dimension:** `4` – each token is represented by a 4-dimensional embedding.

In [ ]:
batch_size, seq_len, embed_dim = 1, 3, 4
x = torch.arange(batch_size * seq_len * embed_dim, dtype=torch.float32)
x = x.view(batch_size, seq_len, embed_dim)

In [ ]:
# important to call the seed every time you create an instance of the attention module.
torch.manual_seed(seed)

model = ScaleDotProductAttention(embed_dim)
attn_output, attn_weights = model(x)

In [ ]:
expected_output = torch.tensor(
    [
        [
            [9.1851, 1.4687, -8.2898, -6.0422],
            [6.8665, 1.0547, -6.1507, -4.3252],
            [4.8920, 0.7021, -4.3290, -2.8630],
        ]
    ],
    dtype=torch.float32,
)

torch.testing.assert_close(attn_output, expected_output, rtol=1e-4, atol=1e-4)

### Task 10: Casual masking in attention

Implement causal (autoregressive) masking in your scaled dot-product attention module so that each token can only attend to itself and previous tokens, not future tokens. This is essential for autoregressive models like GPT, where predicting the next token should not use information from future positions.

In [ ]:
class CasualAttention(nn.Module):
    """
    A simple implementation of scaled dot-product attention with optional causal masking.
    """

    def __init__(self, embed_dim):
        super().__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.scale = embed_dim**0.5

    def forward(self, x):
        """
        x: Tensor of shape (batch, seq_len, embed_dim)
        causal_mask: if True, prevents attention to future tokens
        Returns: output (batch, seq_len, embed_dim), attn_weights (batch, seq_len, seq_len)
        """
        batch_size, seq_len, embed_dim = x.size()

        # Project inputs
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Raw attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        # Create lower triangular mask (1 where allowed, 0 where blocked)
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        # Set disallowed positions to very negative value before softmax
        attn_scores = attn_scores.masked_fill(mask == 0, float("-inf"))

        # Attention weights
        attn_weights = F.softmax(attn_scores, dim=-1)

        # Weighted sum
        attn_output = torch.matmul(attn_weights, V)

        return attn_output, attn_weights

In [ ]:
attn = CasualAttention(embed_dim)

attn_output, attn_weights = attn(x)

In [ ]:
attn_weights

tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00],
         [9.9209e-01, 7.9093e-03, 0.0000e+00],
         [9.9918e-01, 8.1904e-04, 6.7137e-07]]], grad_fn=<SoftmaxBackward0>)

In [ ]:
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().unsqueeze(0)
assert torch.all(attn_weights[mask] < 1e-6)

### Task 12: Transformer Block

Given attention implementation from the previous task is correct make sure the code below works as is.

In [ ]:
class TransformerBlock(nn.Module):
    """A simple Transformer block with attention + feedforward network."""

    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.attn = CasualAttention(embed_dim)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

        # Feedforward network
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, x):
        # Attention + residual
        attn_out, attn_weights = self.attn(self.ln1(x))
        x = x + attn_out

        # Feedforward + residual
        ff_out = self.ff(self.ln2(x))
        x = x + ff_out

        return x, attn_weights

In [ ]:
# Dummy input: batch=1, seq_len=5, embed_dim=16
x = torch.randn(1, 5, 16)

block = TransformerBlock(embed_dim=16, hidden_dim=32)

out, attn_w = block(x)

print("Output shape:", out.shape)  # (1, 5, 16)
print("Attention weights shape:", attn_w.shape)  # (1, 5, 5)

Output shape: torch.Size([1, 5, 16])
Attention weights shape: torch.Size([1, 5, 5])


### Task 13: Code LLM arhitecture

In [ ]:
class LLM(nn.Module):
    """
    A very simple GPT-style language model.
    """

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, max_seq_len):
        super().__init__()
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len

        # Token + positional embeddings
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)

        # Stack of Transformer blocks
        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, hidden_dim) for _ in range(num_layers)]
        )

        # Final layer norm
        self.ln_f = nn.LayerNorm(embed_dim)

        # Output projection to vocabulary
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, input_ids):
        """
        input_ids: Tensor of shape (batch, seq_len)
        Returns: logits (batch, seq_len, vocab_size)
        """
        batch_size, seq_len = input_ids.size()
        device = input_ids.device

        # Token + position embeddings
        token_embeddings = self.token_emb(input_ids)  # (batch, seq_len, embed_dim)
        pos_ids = torch.arange(seq_len, device=device).unsqueeze(0)  # (1, seq_len)
        pos_embeddings = self.pos_emb(pos_ids)  # (1, seq_len, embed_dim)

        x = token_embeddings + pos_embeddings

        # Pass through Transformer blocks
        for block in self.blocks:
            x, _ = block(x)

        # Final norm + vocab projection
        x = self.ln_f(x)
        logits = self.head(x)  # (batch, seq_len, vocab_size)

        return logits

In [ ]:
# Tiny vocab + model
vocab_size = 20
embed_dim = 16
hidden_dim = 32
num_layers = 2
max_seq_len = 10

model = LLM(vocab_size, embed_dim, hidden_dim, num_layers, max_seq_len)

# Dummy batch of token IDs
input_ids = torch.randint(0, vocab_size, (2, 5))  # (batch=2, seq_len=5)

logits = model(input_ids)

print("Input shape:", input_ids.shape)  # (2, 5)
print("logits shape:", logits.shape)  # (2, 5)

Input shape: torch.Size([2, 5])
logits shape: torch.Size([2, 5, 20])
